In [ ]:
"""
One person tells a neighbor (local improvement)
That neighbor tells another
Eventually the whole network knows (global convergence)

Bellman-Ford works because repeated local improvements propagate information through the graph until the full global picture emerges.
"""

In [1]:
import math

def initialize_single_source(vertices, source):
    dist = {v: math.inf for v in vertices}
    pred = {v: None for v in vertices}
    dist[source] = 0
    return dist, pred

def relax(u, v, weight, dist, pred):
    if dist[u] + weight < dist[v]:
        dist[v] = dist[u] + weight
        pred[v] = u

def bellman_ford(vertices, edges, source):
    dist, pred = initialize_single_source(vertices, source)

    for _ in range(len(vertices) - 1):
        for (u, v, w) in edges:
            relax(u, v, w, dist, pred)

    # negative-weight cycles
    for (u, v, w) in edges:
        if dist[u] + w < dist[v]:
            return False, None, None

    return True, dist, pred

In [2]:
vertices = ['A', 'B', 'C', 'D']
edges = [
    ('A', 'B', 1),
    ('B', 'C', 3),
    ('A', 'C', 10),
    ('C', 'D', -2),
    ('D', 'B', -1)
]

ok, dist, pred = bellman_ford(vertices, edges, 'A')

if ok:
    print("Distances:", dist)
    print("Predecessors:", pred)
else:
    print("Graph contains a negative-weight cycle")

Distances: {'A': 0, 'B': 1, 'C': 4, 'D': 2}
Predecessors: {'A': None, 'B': 'A', 'C': 'B', 'D': 'C'}


In [6]:
"""
Modify Bellman-Ford to terminate early if a full pass over all edges performs no relaxations.
Since shortest paths have at most m edges, the algorithm will stop after at most m+1 passes.
"""
import math

def bellman_ford(vertices, edges, source):
    dist = {v: math.inf for v in vertices}
    pred = {v: None for v in vertices}
    dist[source] = 0

    for _ in range(len(vertices) - 1):
        updated = False

        for (u, v, w) in edges:
            if dist[u] + w < dist[v]:
                dist[v] = dist[u] + w
                pred[v] = u
                updated = True

        if not updated:
            break   # EARLY TERMINATION
            
    return dist, pred

In [7]:
"""
After the standard Bellman-Ford relaxation, perform an additional pass
to identify vertices whose distances can still be improved, mark them as part
of or affected by a negative cycle, and propagate −∞ from these vertices to all
reachable vertices.
"""
def bellman_ford(vertices, edges, source):
    dist = {v: float('inf') for v in vertices}
    pred = {v: None for v in vertices}

    dist[source] = 0

    for _ in range(len(vertices) - 1):
        for (u, v, w) in edges:
            if dist[u] + w < dist[v]:
                dist[v] = dist[u] + w
                pred[v] = u

    affected = set()

    for (u, v, w) in edges:
        if dist[u] + w < dist[v]:
            affected.add(v)

    from collections import deque

    queue = deque(affected)

    while queue:
        u = queue.popleft()

        if dist[u] != float('-inf'):
            dist[u] = float('-inf')

        for (a, b, w) in edges:
            if a == u and dist[b] != float('-inf'):
                dist[b] = float('-inf')
                queue.append(b)

    return dist, pred

In [8]:
"""
Bellman-Ford normally computes:
shortest paths FROM one source s

But here we want:
shortest paths TO v from ANY source

Instead of trying every possible source, we simulate "all sources at once"
by adding a fake node that connects to everything for free.

Add a new source vertex s* and connect it to every vertex v ∈ V with an edge of weight 0.
Then run Bellman-Ford with s* as the source. The resulting distances d[v] equal δ*(v) for all v.
The running time is O(VE).
"""
def compute_delta_star(vertices, edges):
    s_star = "super_source"

    new_edges = edges.copy()
    for v in vertices:
        new_edges.append((s_star, v, 0))

    dist = {v: float('inf') for v in vertices}
    dist[s_star] = 0

    for _ in range(len(vertices)):
        for (u, v, w) in new_edges:
            if dist.get(u, float('inf')) + w < dist.get(v, float('inf')):
                dist[v] = dist[u] + w

    return {v: dist[v] for v in vertices}

In [10]:
"""
The algorithm does not return to the source
It returns a closed loop of vertices forming a negative cycle

The loop is found by:
detecting a bad vertex
walking into the cycle
looping until you return to the start

You follow predecessor pointers until they repeat because repetition means you've found a cycle.
"""
def find_negative_cycle(vertices, edges, source):
    dist = {v: float('inf') for v in vertices}
    pred = {v: None for v in vertices}
    dist[source] = 0

    for _ in range(len(vertices) - 1):
        for (u, v, w) in edges:
            if dist[u] + w < dist[v]:
                dist[v] = dist[u] + w
                pred[v] = u

    # negative cycle
    x = None
    for (u, v, w) in edges:
        if dist[u] + w < dist[v]:
            x = v
            break

    if x is None:
        return None  # No negative cycle

    for _ in range(len(vertices)):
        x = pred[x]

    cycle = []
    start = x

    while True:
        cycle.append(x)
        x = pred[x]
        if x == start:
            cycle.append(x)
            break

    cycle.reverse()
    return cycle

In [11]:
def topological_sort(G):
    visited = set()
    stack = []

    def dfs(u):
        visited.add(u)
        for v in G[u]:
            if v not in visited:
                dfs(v)
        stack.append(u)

    for u in G:
        if u not in visited:
            dfs(u)

    return stack[::-1]  # reverse to get correct order

In [12]:
def dag_shortest_paths(G, w, s):
    topo_order = topological_sort(G)
    
    d = {v: float('inf') for v in G}
    pi = {v: None for v in G}
    d[s] = 0
    
    for u in topo_order:
        for v in G[u]:
            if d[v] > d[u] + w[(u, v)]:
                d[v] = d[u] + w[(u, v)]
                pi[v] = u
    
    return d, pi

In [13]:
G = {
    'r': ['s', 't'],
    's': ['t', 'x'],
    't': ['x', 'y', 'z'],
    'x': ['y', 'z'],
    'y': ['z'],
    'z': []
}

w = {
    ('r','s'): 5, ('r','t'): 3,
    ('s','t'): 2, ('s','x'): 6,
    ('t','x'): 7, ('t','y'): 4, ('t','z'): 2,
    ('x','y'): -1, ('x','z'): 1,
    ('y','z'): -2
}

d, pi = dag_shortest_paths(G, w, 'r')

print("Distances:", d)
print("Predecessors:", pi)

Distances: {'r': 0, 's': 5, 't': 3, 'x': 10, 'y': 7, 'z': 5}
Predecessors: {'r': None, 's': 'r', 't': 'r', 'x': 't', 'y': 't', 'z': 't'}


In [15]:
"""
Tasks = vertices
Dependencies = edges
Output = minimum time to finish the project

We flipped shortest-path into longest-path to find the bottleneck chain of tasks,
which determines total completion time.
"""
def dag_longest_paths(G, weight, s):
    topo_order = topological_sort(G)
    
    d = {v: float('-inf') for v in G}
    pi = {v: None for v in G}
    d[s] = weight[s]
    
    for u in topo_order:
        for v in G[u]:
            if d[v] < d[u] + weight[v]:   # MAX instead of MIN
                d[v] = d[u] + weight[v]
                pi[v] = u
    
    return d, pi

In [16]:
G = {
    'A': ['B', 'C'],
    'B': ['D'],
    'C': ['D'],
    'D': ['E'],
    'E': []
}

weight = {
    'A': 3,
    'B': 2,
    'C': 4,
    'D': 2,
    'E': 1
}

source = 'A'

print(dag_longest_paths(G, weight, source))

({'A': 3, 'B': 5, 'C': 7, 'D': 9, 'E': 10}, {'A': None, 'B': 'A', 'C': 'A', 'D': 'C', 'E': 'D'})


In [17]:
"""
Use reverse topological DP where each node counts all paths starting from it.

This is not summing weights it's summing recursively computed path counts,
which encode exponentially many paths.
"""
def count_all_paths(G):
    topo_order = topological_sort(G)
    
    dp = {u: 0 for u in G}
    
    for u in reversed(topo_order):
        dp[u] = 1
        for v in G[u]:
            dp[u] += dp[v]
    
    total_paths = sum(dp.values())
    
    return total_paths

In [18]:
"""
Lazy update (what Python actually does)
heapq does NOT support decrease-key.

So instead:
When you find a better distance, you push a new entry
You don't remove the old one
Later, when popping, you ignore outdated entries

Lazy update = We allow duplicates and clean them up later
Decrease-key = We surgically update the existing entry
"""
import heapq

def dijkstra(G, w, s):
    dist, pred = initialize_single_source(G['V'], s)

    Q = []
    heapq.heappush(Q, (0, s))

    while Q:
        current_dist, u = heapq.heappop(Q)

        if current_dist > dist[u]:
            continue

        for v in G['adj'][u]:
            if relax(u, v, w, dist, pred):
                heapq.heappush(Q, (dist[v], v))

    return dist, pred

In [19]:
G = {
    'V': ['A', 'B', 'C', 'D'],
    'adj': {
        'A': ['B', 'C'],
        'B': ['C', 'D'],
        'C': ['D'],
        'D': []
    }
}

w = {
    ('A', 'B'): 1,
    ('A', 'C'): 4,
    ('B', 'C'): 2,
    ('B', 'D'): 5,
    ('C', 'D'): 1
}



In [20]:
"""
Dijkstra can lock in wrong values early because it assumes no future improvement.

Dijkstra (greedy)
Always picks closest unvisited node

Assumes:
Once I pick a node, its distance is FINAL
Works only if all weights ≥ 0

Bellman-Ford (systematic)
Repeatedly relaxes ALL edges
Does this |V|-1 times
Works even with negative weights

Dijkstra
I trust my current best choice will never improve later

Bellman-Ford
I don't trust anything, I'll keep checking everything repeatedly
"""

"\nDijkstra can lock in wrong values early because it assumes no future improvement.\n\nDijkstra (greedy)\nAlways picks closest unvisited node\n\nAssumes:\nOnce I pick a node, its distance is FINAL\nWorks only if all weights ≥ 0\n\nBellman-Ford (systematic)\nRepeatedly relaxes ALL edges\nDoes this |V|-1 times\nWorks even with negative weights\n\nDijkstra\nI trust my current best choice will never improve later\n\nBellman-Ford\nI don't trust anything, I'll keep checking everything repeatedly\n"

In [21]:
"""
| BFS                 | Modified Dijkstra         |
| ------------------- | ------------------------- |
| Queue (FIFO)        | Priority queue (min-heap) |
| Add when discovered | Add when relaxed          |
| No duplicates       | May have duplicates       |

More efficient for sparse reachable regions
Avoids inserting unreachable vertices
Cleaner mental model (closer to BFS)

You can safely restrict Q to only discovered vertices,
as long as you insert vertices when they are first relaxed and maintain the min-priority property.
"""
import heapq
import math

def dijkstra(G, w, s):
    dist = {v: math.inf for v in G['V']}
    pred = {v: None for v in G['V']}

    dist[s] = 0

    Q = []
    heapq.heappush(Q, (0, s))

    visited = set() # S (S is set that we refer to in CLRS book)

    while Q:
        d_u, u = heapq.heappop(Q)

        if u in visited:
            continue

        visited.add(u)

        for v in G['adj'][u]:
            if dist[v] > dist[u] + w[(u, v)]:
                dist[v] = dist[u] + w[(u, v)]
                pred[v] = u
                heapq.heappush(Q, (dist[v], v))

    return dist, pred

In [22]:
"""
Dijkstra explores vertices in order of current best distance
That order can include vertices not on the shortest path
So relaxations along the shortest path can be interleaved or delayed
"""

'\nDijkstra explores vertices in order of current best distance\nThat order can include vertices not on the shortest path\nSo relaxations along the shortest path can be interleaved or delayed\n'

In [23]:
"""
Maximizing a product = minimizing the negative log of that product
we can transform weights by minimizing the negative log of that product

Since 0 <= r(u,v) <= 1, we have:
log(r) <= 0
so −log(r) >= 0

All edge weights are nonnegative
That means we can safely use Dijkstra

If you want the actual probability:
reliability = math.exp(-dist[t])


Why maximizing becomes minimizing

For a path P:
r(P) = ∏r(u,v)

Take logs:
logr(P) = ∑logr(u,v)

Now multiply by -1:
−logr(P) = ∑−logr(u,v)

So:
maximizing r(P)
is equivalent to minimizing −logr(P)

argmax product : argmin sum of transformed weights

−log doesn't distort the problem it converts multiplicative probabilities
into additive costs so Dijkstra can solve it, and exponentiating restores the original probability.
"""
import math
import heapq

def most_reliable_path(G, r, s, t):
    w = {(u,v): -math.log(r[(u,v)]) for (u,v) in r}  # transform weights

    # Step 2: Dijkstra
    dist = {v: float('inf') for v in G['V']}
    pred = {v: None for v in G['V']}
    dist[s] = 0

    Q = [(0, s)]

    while Q:
        d_u, u = heapq.heappop(Q)

        if u == t:
            break

        if d_u > dist[u]:
            continue

        for v in G['adj'][u]:
            if dist[v] > dist[u] + w[(u,v)]:
                dist[v] = dist[u] + w[(u,v)]
                pred[v] = u
                heapq.heappush(Q, (dist[v], v))

    return dist, pred

In [24]:
"""
using buckets instead of a heap
O(WV + E)

Each edge relaxed once → O(E)
Each bucket scanned at most once → O(WV)

Normal Dijkstra:
Find the smallest distance using a heap

Bucket version:
I already know distances are small integers, just index directly


s → a (1)
s → b (2)
a → c (2)
b → c (1)
c → t (1)

W⋅∣V∣ = 2*5 = 10

We create:
B[0], B[1], B[2], ..., B[10]
Each bucket holds vertices with that exact distance.

Step 0: B[0]: [s]

Step 1:
B[1]: [a]
B[2]: [b]

Step 2:
B[2]: [b]
B[3]: [c]

Step 3:
B[3]: [c]

Step 4:
B[4]: [t]
s:0, a:1, b:2, c:3, t:4

We place a vertex into a bucket when we discover (or improve) its shortest known distance.
Think of it like this

Buckets are indexed by distance:
B[k] = all vertices whose current best distance is k

whenever we discover a path of length k to a node, we drop it into bucket k

You don't pre-fill bins
You drop a letter into bin k when you know its priority is k
"""
def dijkstra_bucket(G, w, s, W):
    import math
    
    dist = {v: math.inf for v in G['V']}
    pred = {v: None for v in G['V']}
    
    dist[s] = 0
    
    # Buckets from 0 to W*|V|
    B = [[] for _ in range(W * len(G['V']) + 1)]
    B[0].append(s)
    
    for i in range(len(B)):
        while B[i]:
            u = B[i].pop()
            
            if dist[u] < i:
                continue  # outdated
            
            for v in G['adj'][u]:
                if dist[v] > dist[u] + w[(u,v)]:
                    old = dist[v]
                    dist[v] = dist[u] + w[(u,v)]
                    pred[v] = u
                    
                    B[dist[v]].append(v)
    
    return dist, pred

In [26]:
"""
By grouping distance estimates into exponentially growing ranges instead of exact values,
we reduce the number of buckets from O(WV) to O(log W), giving O((V+E) log W) time.

| Method                 | Structure                  | Complexity     |
| ---------------------- | -------------------------- | -------------- |
| Simple bucket Dijkstra | exact distance buckets     | O(WV + E)      |
| buket ranges w/ heaps  | logarithmic buckets + heap | O((V+E) log W) |
| standard Dijkstra      | binary heap                | O((V+E) log V) |

Level i covers: [2^i, 2^(i+1) − 1]

| Level | Range  |
| ----- | ------ |
| 0     | [0]    |
| 1     | [1]    |
| 2     | [2–3]  |
| 3     | [4–7]  |
| 4     | [8–15] |
| ...   | ...    |

Each level doubles the range:

Level 1: very precise (1 value)
Level 2: small range (2 values)
Level 3: bigger range (4 values)
Level 4: even bigger (8 values)

So we "zoom out" gradually.

If we used:
[0–3], [4–7], [8–11], ...

That would still be O(W), not O(log W).

We want:
fewer and fewer buckets as values grow

Why powers of 2 specifically?

Because they give:
clean partitioning
logarithmic number of levels
easy indexing via bit length

In fact:
bucket level ≈ floor(log2(distance))

Instead of asking:
What exact number is this?

We ask:
How many bits do I need to represent it?

That's why:
1–1 → small bucket
2–3 → next bucket
4–7 → next
8–15 → next
"""

'\nBy grouping distance estimates into exponentially growing ranges instead of exact values,\nwe reduce the number of buckets from O(WV) to O(log W), giving O((V+E) log W) time.\n\n| Method                 | Structure                  | Complexity     |\n| ---------------------- | -------------------------- | -------------- |\n| Simple bucket Dijkstra | exact distance buckets     | O(WV + E)      |\n| buket ranges w/ heaps  | logarithmic buckets + heap | O((V+E) log W) |\n| standard Dijkstra      | binary heap                | O((V+E) log V) |\n'

In [27]:
"""
We maintain buckets indexed by:
distance mod K

Data structure:
array of buckets of size 2C (constant)
circular scanning pointer

distances are tightly bounded and grow smoothly
So we can replace the heap with a constant-size rotating structure.

If weights were arbitrary:
distances explode
buckets become huge → O(WV)

But here:
max/min weight ratio = 2
So growth is controlled.

Because all edge weights lie in a constant-sized interval [C, 2C],
we can replace the priority queue with a constant number of rotating buckets, yielding O(V + E) time.
"""
def dijkstra_fast(G, s, C):
    dist = {v: float('inf') for v in G.V}
    dist[s] = 0

    B = [[] for _ in range(2*C + 1)]
    B[0].append(s)

    i = 0  # current bucket index

    while True:
        while i < len(B) and not B[i]:
            i = (i + 1) % len(B)

        if i >= len(B):
            break

        u = B[i].pop()

        for v in G.adj[u]:
            w = weight(u,v)

            if dist[v] > dist[u] + w:
                old_bucket = dist[v] % len(B) if dist[v] != float('inf') else None
                dist[v] = dist[u] + w
                B[dist[v] % len(B)].append(v)

    return dist

In [28]:
"""
DIFFERENCE CONSTRAINTS → GRAPH (Bellman-Ford)

Standard form:
    xi - xj ≤ c

Rewrite:
    xi ≤ xj + c

Graph rule:
    Add edge: j → i with weight c

Meaning:
    xi depends on xj
    So the arrow goes FROM the right variable TO the left variable

----------------------------------------

EXAMPLES

x4 - x5 ≤ 10
→ x4 ≤ x5 + 10
→ edge: 5 → 4 (weight 10)

x2 - x4 ≤ -6
→ x2 ≤ x4 - 6
→ edge: 4 → 2 (weight -6)

x5 - x3 ≤ -4
→ x5 ≤ x3 - 4
→ edge: 3 → 5 (weight -4)

----------------------------------------

FULL ALGORITHM

1. Create one node per variable (x1, x2, ..., xn)

2. Add edges using rule:
       xi - xj ≤ c  ⇒  j → i (weight c)

3. Add super-source s:
       s → xi (weight 0 for all i)

4. Run Bellman-Ford from s

5. If negative cycle exists:
       → NO feasible solution

6. Otherwise:
       → solution exists
       → xi = distance[i]

----------------------------------------

INTUITION

Each constraint says:
    "you can go from xj to xi with cost c"

If you can loop and get a NEGATIVE total cost:
    → contradiction → impossible system
"""

'\nDIFFERENCE CONSTRAINTS → GRAPH (Bellman-Ford)\n\nStandard form:\n    xi - xj ≤ c\n\nRewrite:\n    xi ≤ xj + c\n\nGraph rule:\n    Add edge: j → i with weight c\n\nMeaning:\n    xi depends on xj\n    So the arrow goes FROM the right variable TO the left variable\n\n----------------------------------------\n\nEXAMPLES\n\nx4 - x5 ≤ 10\n→ x4 ≤ x5 + 10\n→ edge: 5 → 4 (weight 10)\n\nx2 - x4 ≤ -6\n→ x2 ≤ x4 - 6\n→ edge: 4 → 2 (weight -6)\n\nx5 - x3 ≤ -4\n→ x5 ≤ x3 - 4\n→ edge: 3 → 5 (weight -4)\n\n----------------------------------------\n\nFULL ALGORITHM\n\n1. Create one node per variable (x1, x2, ..., xn)\n\n2. Add edges using rule:\n       xi - xj ≤ c  ⇒  j → i (weight c)\n\n3. Add super-source s:\n       s → xi (weight 0 for all i)\n\n4. Run Bellman-Ford from s\n\n5. If negative cycle exists:\n       → NO feasible solution\n\n6. Otherwise:\n       → solution exists\n       → xi = distance[i]\n\n----------------------------------------\n\nINTUITION\n\nEach constraint says:\n    "you can

In [30]:
"""
BELOW (explicit)
Add node s
Add edges (s → xi, 0)
Set: d(s) = 0, d(xi) = ∞

Shortcut way
No s
Just set: d(xi) = 0∀i

👉 These produce the same effect:

Every node starts with a reachable path of cost 0
"""
# VARIABLES
vertices = ['s', 'x1', 'x2', 'x3', 'x4', 'x5']

# EDGES (from constraints xi - xj ≤ c  ⇒  j → i with weight c)
# SUPER-SOURCE: This enforces: xi ≤ xs +0 (constraint)
edges = [
    # From system (22.4-2)
    ('x2', 'x1', 4),     # x1 - x2 ≤ 4
    ('x5', 'x1', 5),     # x1 - x5 ≤ 5
    ('x4', 'x2', -6),    # x2 - x4 ≤ -6
    ('x2', 'x3', 1),     # x3 - x2 ≤ 1
    ('x1', 'x4', 3),     # x4 - x1 ≤ 3
    ('x3', 'x4', 5),     # x4 - x3 ≤ 5
    ('x5', 'x4', 10),    # x4 - x5 ≤ 10
    ('x3', 'x5', -4),    # x5 - x3 ≤ -4
    ('x4', 'x5', -8),    # x5 - x4 ≤ -8

    # Super-source edges
    ('s', 'x1', 0),
    ('s', 'x2', 0),
    ('s', 'x3', 0),
    ('s', 'x4', 0),
    ('s', 'x5', 0),
]

# RUN
dist, pred = bellman_ford(vertices, edges, 's')

print("dist:", dist)
print("pred:", pred)

dist: {'s': 0, 'x1': -inf, 'x2': -inf, 'x3': -inf, 'x4': -inf, 'x5': -inf}
pred: {'s': None, 'x1': 'x5', 'x2': 'x4', 'x3': 'x2', 'x4': 'x1', 'x5': 'x3'}


In [32]:
"""
Push values downward until all constraints are satisfied.

Start with everything at 0
If a constraint requires something smaller, decrease it
Keep propagating

This works even if the graph is disconnected.

Why?

In standard Bellman–Ford, unreachable nodes stay at +∞
Here, all nodes start at 0 → effectively "reachable"

So:
No need for a source
No need for connectivity

Example

Constraint:
x2 − x1 ≤ − 3

Edge:
1→2 with weight −3

Initialize:
d(1) = d(2) = 0

Relax:
d(2) = min(0,0−3) = −3

So:
x2 = −3, x1 = 0

ABOVE:
d[s] = 0
d[x_i] = inf
run Bellman-Ford from s
(removing):
# Super-source edges
('s', 'x1', 0),
('s', 'x2', 0),
('s', 'x3', 0),
('s', 'x4', 0),
('s', 'x5', 0),

WITHOUT SUPER-SOURCE (these notes):
for all x_i:
    d[x_i] = 0

run Bellman-Ford on all edges

Same feasibility test (negative cycle detection)
Same relative solution structure
Different absolute offsets (which don’t matter in difference constraints)
"""

'\nPush values downward until all constraints are satisfied.\n\nStart with everything at 0\nIf a constraint requires something smaller, decrease it\nKeep propagating\n\nThis works even if the graph is disconnected.\n\nWhy?\n\nIn standard Bellman–Ford, unreachable nodes stay at +∞\nHere, all nodes start at 0 → effectively "reachable"\n\nSo:\nNo need for a source\nNo need for connectivity\n\nExample\n\nConstraint:\nx2 − x1 ≤ − 3\n\nEdge:\n1→2 with weight −3\n\nInitialize:\nd(1) = d(2) = 0\n\nRelax:\nd(2) = min(0,0−3) = −3\n\nSo:\nx2 = −3, x1 = 0\n\nABOVE:\nd[s] = 0\nd[x_i] = inf\nrun Bellman-Ford from s\n\nWITHOUT SUPER-SOURCE (these notes):\nfor all x_i:\n    d[x_i] = 0\n\nrun Bellman-Ford on all edges\n'

In [34]:
"""
We want to solve a system of constraints of the form:

1) Difference constraints:
    x_i - x_j ≤ c
    → edge: (j → i) with weight c

2) Upper bounds:
    x_i ≤ b
    → rewrite: x_i ≤ 0 + b
    → edge: (s → i) with weight b

3) Lower bounds:
    -x_i ≤ b   ⇔   x_i ≥ -b
    → rewrite: 0 ≤ x_i + b   ⇔   0 - x_i ≤ b
    → edge: (i → s) with weight b

Bellman-Ford invariant (what the algorithm enforces):
    d(v) ≤ d(u) + w(u, v)

This matches each constraint exactly:
    x_i ≤ x_j + c     ← (j → i, weight c)
    x_i ≤ 0 + b       ← (s → i, weight b)
    0 ≤ x_i + b       ← (i → s, weight b)


Interpretation:
- edges from s = upper bounds (push values downward)
- edges to s   = lower bounds (push values upward)
- edges between variables = propagate constraints

Upper bounds:    s → x_i     → push x_i DOWN
Lower bounds:    x_i → s     → push x_i UP
Differences:     x_j → x_i   → tie variables together

Bellman-Ford finds equilibrium where:
- all constraints are satisfied
- no value can be reduced further
"""

'\nWe want to solve a system of constraints of the form:\n\n1) Difference constraints:\n    x_i - x_j ≤ c\n    → edge: (j → i) with weight c\n\n2) Upper bounds:\n    x_i ≤ b\n    → rewrite: x_i ≤ 0 + b\n    → edge: (s → i) with weight b\n\n3) Lower bounds:\n    -x_i ≤ b   ⇔   x_i ≥ -b\n    → rewrite: 0 ≤ x_i + b   ⇔   0 - x_i ≤ b\n    → edge: (i → s) with weight b\n\nBellman-Ford invariant (what the algorithm enforces):\n    d(v) ≤ d(u) + w(u, v)\n\nThis matches each constraint exactly:\n    x_i ≤ x_j + c     ← (j → i, weight c)\n    x_i ≤ 0 + b       ← (s → i, weight b)\n    0 ≤ x_i + b       ← (i → s, weight b)\n\n\nInterpretation:\n- edges from s = upper bounds (push values downward)\n- edges to s   = lower bounds (push values upward)\n- edges between variables = propagate constraints\n\nUpper bounds:    s → x_i     → push x_i DOWN\nLower bounds:    x_i → s     → push x_i UP\nDifferences:     x_j → x_i   → tie variables together\n\nBellman-Ford finds equilibrium where:\n- all constr

In [35]:
"""
Initially, d(s) = 0 and s.π = NIL.

If s.π becomes non-NIL during relaxation, then some edge (u, s) must have been relaxed, implying:
    d(s) > d(u) + w(u, s)

Since d(s) = 0, we have:
    0 > d(u) + w(u, s)

By properties of relaxation, d(u) corresponds to the weight of some path from s to u. Thus, there exists a path from s to u of weight d(u). Adding edge (u, s) forms a cycle whose total weight is:
    d(u) + w(u, s) < 0

Therefore, the graph contains a negative-weight cycle.
"""

'\nInitially, d(s) = 0 and s.π = NIL.\n\nIf s.π becomes non-NIL during relaxation, then some edge (u, s) must have been relaxed, implying:\n    d(s) > d(u) + w(u, s)\n\nSince d(s) = 0, we have:\n    0 > d(u) + w(u, s)\n\nBy properties of relaxation, d(u) corresponds to the weight of some path from s to u. Thus, there exists a path from s to u of weight d(u). Adding edge (u, s) forms a cycle whose total weight is:\n    d(u) + w(u, s) < 0\n\nTherefore, the graph contains a negative-weight cycle.\n'

In [36]:
"""
Since the graph contains no negative-weight cycles, every shortest path is simple and contains at most |V| - 1 edges.

We construct a sequence of relaxations by processing edges in order of increasing shortest-path distance (measured in number of edges from s). Specifically, we relax all edges on shortest paths of length 1 first, then length 2, and so on up to length |V| - 1.

We prove by induction on k that after processing all edges on shortest paths of at most k edges, we have v.d = δ(s, v) for all such vertices v.

Base case: For k = 0, only s is included and INITIALIZE-SINGLE-SOURCE gives s.d = 0 = δ(s, s).

Inductive step: Let v be a vertex whose shortest path has k edges, and let (u, v) be the last edge on a shortest path to v. By induction, u.d = δ(s, u). Relaxing (u, v) yields:
v.d ≤ u.d + w(u, v) = δ(s, u) + w(u, v) = δ(s, v).
Since distances are never underestimated, v.d = δ(s, v).

Thus, after |V| - 1 stages, all vertices satisfy v.d = δ(s, v).
"""

'\nSince the graph contains no negative-weight cycles, every shortest path is simple and contains at most |V| - 1 edges.\n\nWe construct a sequence of relaxations by processing edges in order of increasing shortest-path distance (measured in number of edges from s). Specifically, we relax all edges on shortest paths of length 1 first, then length 2, and so on up to length |V| - 1.\n\nWe prove by induction on k that after processing all edges on shortest paths of at most k edges, we have v.d = δ(s, v) for all such vertices v.\n\nBase case: For k = 0, only s is included and INITIALIZE-SINGLE-SOURCE gives s.d = 0 = δ(s, s).\n\nInductive step: Let v be a vertex whose shortest path has k edges, and let (u, v) be the last edge on a shortest path to v. By induction, u.d = δ(s, u). Relaxing (u, v) yields:\nv.d ≤ u.d + w(u, v) = δ(s, u) + w(u, v) = δ(s, v).\nSince distances are never underestimated, v.d = δ(s, v).\n\nThus, after |V| - 1 stages, all vertices satisfy v.d = δ(s, v).\n'

In [37]:
"""
The idea is exactly about preventing exploitable loopholes, but the reality is a bit more subtle.

If exchange rates were perfectly consistent, then:

No cycle of trades would ever give profit
Every product around a cycle would equal exactly 1

But real markets are:
decentralized
slightly delayed between updates
affected by fees, spreads, and liquidity differences

So temporary inconsistencies can exist.

In practice (modern markets)
A → B → C → A

does happen, but:
It is usually extremely small (fractions of a cent)
It exists for milliseconds or less
It is quickly removed by automated trading systems

Not central banks directly but:
High-frequency trading firms
Forex algorithmic traders
Crypto trading bots

They continuously run models equivalent to:
"find negative cycles in exchange-rate graphs"
using ideas very close to Bellman-Ford, but much faster specialized systems.

Math model: perfect, clean graph → arbitrage = negative cycle
Real world: noisy, fast-moving graph → arbitrage = rare, tiny, fleeting

Arbitrage detection can be modeled using a graph transformation with logarithms.

Example:
Suppose we have three currencies A, B, and C with exchange rates:
A → B = 2
B → C = 3
C → A = 0.2

Starting with 1 unit of A:
1 × 2 × 3 × 0.2 = 1.2 > 1
So there is a profitable arbitrage cycle.

Now transform using negative logarithms:
w(i, j) = -log(R(i, j))

Compute weights:
A → B: -log(2) ≈ -0.693
B → C: -log(3) ≈ -1.099
C → A: -log(0.2) ≈ 1.609

Now sum around the cycle:
-0.693 + (-1.099) + 1.609 = -0.183 < 0

Thus:
A product greater than 1 corresponds exactly to a negative-weight cycle after the log transformation.

This reduces arbitrage detection to finding negative cycles in a graph, which can be done efficiently using the Bellman-Ford algorithm with edge weights w(i,j) = -log(R(i,j)).
"""

'\nThe idea is exactly about preventing exploitable loopholes, but the reality is a bit more subtle.\n\nIf exchange rates were perfectly consistent, then:\n\nNo cycle of trades would ever give profit\nEvery product around a cycle would equal exactly 1\n\nBut real markets are:\ndecentralized\nslightly delayed between updates\naffected by fees, spreads, and liquidity differences\n\nSo temporary inconsistencies can exist.\n\nIn practice (modern markets)\nA → B → C → A\n\ndoes happen, but:\nIt is usually extremely small (fractions of a cent)\nIt exists for milliseconds or less\nIt is quickly removed by automated trading systems\n\nNot central banks directly but:\nHigh-frequency trading firms\nForex algorithmic traders\nCrypto trading bots\n\nThey continuously run models equivalent to:\n"find negative cycles in exchange-rate graphs"\nusing ideas very close to Bellman-Ford, but much faster specialized systems.\n\nMath model: perfect, clean graph → arbitrage = negative cycle\nReal world: nois

In [38]:
"""
How to speed up shortest path when weights are bounded integers
How to combine:
relaxation ideas (Bellman-Ford/Dijkstra style)
bit decomposition (like divide-and-conquer over digits)

Think of shortest paths like this:
True weights are big numbers
But we solve them in layers of precision

Full weight:  101011 (binary)
Phase 1:      100000  → coarse solution
Phase 2:      101000  → refined
Phase 3:      101011  → exact

Each phase fixes only a “chunk” of the answer.

So instead of solving once with heavy work:
we solve multiple cheap approximations

| Phase | What you see         |
| ----- | -------------------- |
| δ₁    | blurry sketch        |
| δ₂    | clearer outline      |
| δ₄    | almost final         |
| δ₈    | exact shortest paths |
"""
import math
from collections import defaultdict

def relax(edges, dist):
    """One pass of edge relaxation"""
    changed = False
    for u, v, w in edges:
        if dist[u] + w < dist[v]:
            dist[v] = dist[u] + w
            changed = True
    return changed


def compute_delta_leq_E(n, edges, source):
    """
    Part (a): assumes all shortest paths ≤ |E|
    Runs O(E) style bounded propagation
    """
    dist = [float('inf')] * n
    dist[source] = 0

    # at most |E| useful propagation rounds, but usually much less
    for _ in range(len(edges)):
        if not relax(edges, dist):
            break

    return dist


def compute_delta_1(n, edges, source):
    """
    Part (b): single-scale shortest paths (δ₁)
    One relaxation pass
    """
    dist = [float('inf')] * n
    dist[source] = 0

    relax(edges, dist)

    return dist

In [40]:
"""
Each phase:
doubles previous solution
fixes a small “error” (0/1 per edge)
solves that error cheaply

So instead of:
solving one big problem expensively

you:
solve many tiny corrections efficiently

In Gabow's scaling algorithm, edge weights are revealed one bit at a time.
Thus each refined weight satisfies:
w_i(u,v) = 2w_{i-1}(u,v) or 2w_{i-1}(u,v) + 1.

This implies the shortest-path distances scale as:
2δ_{i-1}(s,v) ≤ δ_i(s,v) ≤ 2δ_{i-1}(s,v) + |V| − 1.

To update efficiently, define reweighted edges:
ŵ_i(u,v) = w_i(u,v) + 2δ_{i-1}(s,u) − 2δ_{i-1}(s,v).

Using the triangle inequality, these reweighted weights are always
nonnegative and in fact only 0 or 1.

The new distances satisfy:
δ_i(s,v) = δ̂_i(s,v) + 2δ_{i-1}(s,v),

where δ̂_i(s,v) is the shortest path under the reweighted graph.

Since all ŵ_i are 0 or 1, δ̂_i can be computed in O(E) time
(using 0-1 BFS), giving a total running time of O(E log W).
"""

'\nEach phase:\ndoubles previous solution\nfixes a small “error” (0/1 per edge)\nsolves that error cheaply\n\nSo instead of:\nsolving one big problem expensively\n\nyou:\nsolve many tiny corrections efficiently\n\nIn Gabow’s scaling algorithm, edge weights are revealed one bit at a time.\nThus each refined weight satisfies:\nw_i(u,v) = 2w_{i-1}(u,v) or 2w_{i-1}(u,v) + 1.\n\nThis implies the shortest-path distances scale as:\n2δ_{i-1}(s,v) ≤ δ_i(s,v) ≤ 2δ_{i-1}(s,v) + |V| − 1.\n\nTo update efficiently, define reweighted edges:\nŵ_i(u,v) = w_i(u,v) + 2δ_{i-1}(s,u) − 2δ_{i-1}(s,v).\n\nUsing the triangle inequality, these reweighted weights are always\nnonnegative and in fact only 0 or 1.\n\nThe new distances satisfy:\nδ_i(s,v) = δ̂_i(s,v) + 2δ_{i-1}(s,v),\n\nwhere δ̂_i(s,v) is the shortest path under the reweighted graph.\n\nSince all ŵ_i are 0 or 1, δ̂_i can be computed in O(E) time\n(using 0-1 BFS), giving a total running time of O(E log W).\n'

In [41]:
"""
(a) Suppose μ* = 0. Then every cycle in G has mean weight ≥ 0.
This implies that every cycle has total weight ≥ 0, so there are
no negative-weight cycles in G.

From shortest path theory (Bellman-Ford), if there are no negative
cycles, then every shortest path can be taken to be simple, and thus
uses at most n − 1 edges.

Therefore:
δ(s,v) = min { δ_k(s,v) : 0 ≤ k ≤ n − 1 }.

---

(b) Assume μ* = 0. From part (a), we know:
1) There are no negative-weight cycles.
2) δ(s,v) = min { δ_k(s,v) : 0 ≤ k ≤ n − 1 }.

Now consider any path from s to v with exactly n edges. Since any
simple path has at most n − 1 edges, a path with n edges must contain
a cycle. Because all cycles have nonnegative weight, removing cycles
cannot increase the path weight.

Thus:
δ_n(s,v) ≥ δ(s,v).

Also from (a):
δ(s,v) ≤ δ_k(s,v) for all k ≤ n − 1.

Combining:
δ_n(s,v) ≥ δ(s,v) ≤ δ_k(s,v)

Rearrange:
δ_n(s,v) − δ_k(s,v) ≥ 0

Divide by (n − k) > 0:
(δ_n(s,v) − δ_k(s,v)) / (n − k) ≥ 0

Therefore:
max { (δ_n(s,v) − δ_k(s,v)) / (n − k) : 0 ≤ k ≤ n − 1 } ≥ 0.
"""

'\n(a) Suppose μ* = 0. Then every cycle in G has mean weight ≥ 0.\nThis implies that every cycle has total weight ≥ 0, so there are\nno negative-weight cycles in G.\n\nFrom shortest path theory (Bellman-Ford), if there are no negative\ncycles, then every shortest path can be taken to be simple, and thus\nuses at most n − 1 edges.\n\nTherefore:\nδ(s,v) = min { δ_k(s,v) : 0 ≤ k ≤ n − 1 }.\n\n---\n\n(b) Assume μ* = 0. From part (a), we know:\n1) There are no negative-weight cycles.\n2) δ(s,v) = min { δ_k(s,v) : 0 ≤ k ≤ n − 1 }.\n\nNow consider any path from s to v with exactly n edges. Since any\nsimple path has at most n − 1 edges, a path with n edges must contain\na cycle. Because all cycles have nonnegative weight, removing cycles\ncannot increase the path weight.\n\nThus:\nδ_n(s,v) ≥ δ(s,v).\n\nAlso from (a):\nδ(s,v) ≤ δ_k(s,v) for all k ≤ n − 1.\n\nCombining:\nδ_n(s,v) ≥ δ(s,v) ≤ δ_k(s,v)\n\nRearrange:\nδ_n(s,v) − δ_k(s,v) ≥ 0\n\nDivide by (n − k) > 0:\n(δ_n(s,v) − δ_k(s,v)) / (n − k

In [42]:
"""
Karp's algorithm turns shortest-path differences across path lengths into exact measurements of cycle averages.

======================
n = 3
source = 0

edges = [
    (0, 1, 1),
    (1, 2, -2),
    (2, 0, 1)
]
======================

short path (k edges)
      ↓
long path (n edges)
      ↓
extra edges = cycles

(δ_n - δ_k) = weight of cycles
(n - k)     = number of edges in cycles
ratio = average cycle weight

Graph:
Vertices = {0, 1, 2}, source s = 0
Edges:
0 → 1 (weight = 1)
1 → 2 (weight = -2)
2 → 0 (weight = 1)

Cycle:
0 → 1 → 2 → 0 has total weight 1 + (-2) + 1 = 0
Mean weight = 0 / 3 = 0

Step 1: Compute δ_k(s,v), shortest path from s to v using exactly k edges

k = 0:
δ₀ = [0, ∞, ∞]

k = 1:
δ₁ = [∞, 1, ∞]

k = 2:
δ₂ = [∞, ∞, -1]

k = 3:
δ₃ = [0, ∞, ∞]

k   δ_k(0)   δ_k(1)   δ_k(2)
--------------------------------
0     0        ∞        ∞
1     ∞        1        ∞
2     ∞        ∞       -1
3     0        ∞        ∞

Step 2: Apply Karp formula for each vertex v:
μ(v) = max over k = 0 to n-1 of (δ_n(s,v) - δ_k(s,v)) / (n - k)

Here n = 3

For v = 0:
k = 0: (0 - 0)/3 = 0
k = 1,2: invalid (∞ values)
So max = 0

Other vertices have δ₃ = ∞, so ignore

Step 3: Take minimum over all vertices:
μ* = 0

DP table = all possible reachability states
Karp formula = only compares compatible endpoints

Karp's algorithm doesn't ignore δₖ values, it only compares them when both
represent valid endpoints for n-edge and k-edge paths to the same vertex.
"""
def karp_min_mean_cycle(n, edges, source):
    INF = float('inf')
    
    delta = [[INF] * n for _ in range(n+1)]
    delta[0][source] = 0
    
    for k in range(1, n+1):
        for u, v, w in edges:
            if delta[k-1][u] != INF:
                delta[k][v] = min(delta[k][v], delta[k-1][u] + w)
    
    mu_star = float('inf')
    
    for v in range(n):
        max_val = -float('inf')
        for k in range(n):
            if delta[n][v] < INF and delta[k][v] < INF:
                val = (delta[n][v] - delta[k][v]) / (n - k)
                max_val = max(max_val, val)
        mu_star = min(mu_star, max_val)
    
    return mu_star

In [44]:
"""
Bitonic constraint converts shortest paths into two monotone DP passes (increasing + decreasing),
making the problem solvable in O(ElogE) instead of general shortest-path complexity.

Avoiding combinatorial explosion of paths.
dynamic programming over a global ordering constraint

Instead of tracking:
exponentially many paths

we track:
just two DP arrays:
inc[v]
dec[v]

Instead of exploring all paths, we use:
"Every shortest path has exactly one peak edge"

So we:
build all possible left sides
build all possible right sides
glue them at the peak

sort edges by weight

# increasing DP from source
for edges in increasing order:
    relax inc[v] using inc[u] + w(u,v)

# decreasing DP (reverse idea)
for edges in decreasing order:
    relax dec[v]

# combine
for each edge (u,v):
    best[v] = min(best[v], inc[u] + w(u,v) + dec[v])

s → A (2)
A → B (5)
B → C (9)   ← peak (largest)
C → D (6)
D → v (1)

2 → 5 → 9 → 6 → 1

        (increase)
s ──2──► A ──5──► B ──9──► C
                           ▲
                           │  peak (maximum weight edge = 9)
                           │
              D ◄──6── C ──┘
              ▲
              │
              1
              │
              v
              
Build all increasing paths to every node
s → A → B → C
2. Build all decreasing paths from every node
C → D → v

(best increasing to C) + (best decreasing from C)

A bitonic path has exactly one turning point:

     peak
      ▲
     / \
    /   \
increasing  decreasing

So instead of exploring exponentially many paths, we only compute:
best ways going up
best ways going down
glue at the peak

A bitonic shortest path is just two monotone shortest-path problems stitched together at the maximum-weight edge.

key analogy

Think of it like this:

Brute force:
try every possible route through a maze

Bitonic algorithm:
precompute:
best way to reach every room while going uphill
best way to leave every room while going downhill
then combine

You still walk every corridor—but only in structured passes, not path-by-path exploration.

Runtime depends on how many times we recompute states, not how many edges exist.

The speedup comes not from reducing edge scanning, but from collapsing exponentially
many paths into a constant number of meaningful summaries per vertex.

BRUTE FORCE VIEW (exploding paths):

            s
          /   \
        A       B
        | \     |
        |  \    |
        C   C   C
       / \ / \ / \
      D   v D  v  v

All possible paths are explored separately:
s→A→C→v
s→A→C→D→v
s→B→C→v
s→B→C→D→v
(same subpaths recomputed many times)


------------------------------------------------------------


BITONIC DP VIEW (compressed structure):

STEP 1: increasing structure (store best inc[v])

        s
       / \
      A   B
       \ /
        C     ← keep ONLY best way to reach C

STEP 2: decreasing structure (store best dec[v])

        C
       / \
      D   v     ← keep ONLY best way from C downward


STEP 3: combine at peak (one stitch point)

        s
       / \
      A   B
       \ /
        C   ← peak edge/node
       / \
      D   v

Final bitonic path is formed as:
increasing: s → ... → C
decreasing: C → ... → v

------------------------------------------------------------

INTUITION:
Brute force = expands into a tree of ALL possible paths
Bitonic DP = collapses everything into:
  - one best "upward summary" per node (inc)
  - one best "downward summary" per node (dec)
  - one merge at the peak
"""

'\nBitonic constraint converts shortest paths into two monotone DP passes (increasing + decreasing),\nmaking the problem solvable in O(ElogE) instead of general shortest-path complexity.\n\nAvoiding combinatorial explosion of paths.\ndynamic programming over a global ordering constraint\n\nInstead of tracking:\nexponentially many paths\n\nwe track:\njust two DP arrays:\ninc[v]\ndec[v]\n\nInstead of exploring all paths, we use:\n"Every shortest path has exactly one peak edge"\n\nSo we:\nbuild all possible left sides\nbuild all possible right sides\nglue them at the peak\n\nsort edges by weight\n\n# increasing DP from source\nfor edges in increasing order:\n    relax inc[v] using inc[u] + w(u,v)\n\n# decreasing DP (reverse idea)\nfor edges in decreasing order:\n    relax dec[v]\n\n# combine\nfor each edge (u,v):\n    best[v] = min(best[v], inc[u] + w(u,v) + dec[v])\n\ns → A (2)\nA → B (5)\nB → C (9)   ← peak (largest)\nC → D (6)\nD → v (1)\n\n2 → 5 → 9 → 6 → 1\n\n        (increase)\ns ──